# The memory of LLM agents — companion notebook

*English version — the original is [`agent_memory_langgraph_deepagents.ipynb`](agent_memory_langgraph_deepagents.ipynb).*

Internal Data Engineering / Data Science workshop — Session 2 (hands-on).

Slide support: `docs/slides/agent-memory-part1-theory.html` and
`docs/slides/agent-memory-part2-practice.html`.

**Outline:**
1. Cosine similarity "from scratch" (numpy)
2. Retrieval score à la *Generative Agents* (recency + importance + relevance)
3. LangGraph — short-term memory (checkpointer, threads, time travel)
4. LangGraph — long-term memory (`Store`, namespaces, semantic search, `langmem`)
5. DeepAgents — sub-agents, virtual filesystem, cross-session persistence

**Important — running with or without an API key:**
Sections 1 and 2 are 100% local (numpy only, no LLM). Sections
3 to 5 use a real Anthropic model if the environment variable
`ANTHROPIC_API_KEY` is set, and automatically fall back to a
mock (deterministic, offline) model otherwise — so everyone on the team
can run the notebook end to end before the session, with or without API access.


## Glossary

| Term | Definition |
|---|---|
| **Harness** (agent harness) | The code wrapped around an LLM to turn it into a full agent: the decision loop, tool handling, memory and context management — the LLM alone doesn't know how to loop or call tools by itself. |
| **Opinionated** | Imposing default design choices rather than leaving everything configurable (the opposite of "unopinionated"/neutral) — a trade-off between getting started fast and staying flexible. |
| **Batteries-included** | Shipped with everything needed to be useful right away, without assembling third-party pieces yourself. |
| **Checkpointer** | The LangGraph component that saves a graph's state at every step so it can be resumed later (= short-term memory). |
| **Store** | LangGraph's namespaced key/value storage component, independent of threads (= long-term memory). |
| **Backend** (DeepAgents) | The system that decides where virtual files actually live (ephemeral state, a persistent Store, disk...). |
| **Sub-agent (quarantine)** | A helper agent running in its own isolated context window; only its final result crosses over to the parent agent. |
| **Offloading** | Moving information out of the active context window (e.g. to a file) to free up token budget. |

Full glossary (with `Token`, `Embedding`, `RAG`, `Framework`, `Model-agnostic`, `Grounding`, `Context engineering`): see the slides, "Glossary" section.


In [ ]:
# Installation (run once, uncomment if needed)
# %pip install -q "langgraph>=1.2,<2.0" "langchain>=1.4,<2.0" "langmem>=0.0.30" \
#     "deepagents>=0.7,<0.8" "langchain-anthropic" numpy

import os
import math
import uuid
import time
from datetime import datetime, timedelta

import numpy as np

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAS_ANTHROPIC = bool(ANTHROPIC_API_KEY)
print("ANTHROPIC_API_KEY detected:", HAS_ANTHROPIC)
if not HAS_ANTHROPIC:
    print("-> Sections 3 to 5 will use an offline mock model for the demo.")


## 1. Cosine similarity, from scratch

Almost every "vector memory" system relies on a single
formula:

$$\cos(u, v) = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}$$

To avoid depending on any API key at this stage, we use a **toy embedding**
(bag-of-words hashing) rather than a real embedding model — the idea remains
strictly the same with real embeddings (OpenAI, Voyage, Cohere, ...).


In [ ]:
def toy_embed(text: str, dims: int = 256) -> np.ndarray:
    """Toy, deterministic embedding (word hashing) — for the offline demo.
    Replace with a real embedding model in production."""
    vec = np.zeros(dims, dtype=float)
    for word in text.lower().split():
        idx = hash(word) % dims
        vec[idx] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denom) if denom > 0 else 0.0


phrases = [
    "the airflow pipeline failed last night",
    "the airflow dag errored out yesterday evening",
    "the client prefers concise answers, code first",
    "apple pie recipe for this weekend",
]

query = "why did the airflow job crash?"
q_vec = toy_embed(query)

print(f"Query: {query!r}\n")
for p in phrases:
    sim = cosine_similarity(q_vec, toy_embed(p))
    print(f"  cos = {sim:0.3f}   {p!r}")


We see the two sentences about the Airflow pipeline standing out clearly above
the other two — this is exactly the mechanism (with real embeddings in
place of the toy hashing) that powers the semantic search of the `Store`
LangGraph in section 4.


## 2. Retrieval score à la *Generative Agents*

Park et al. (2023) combine three signals to decide which memory to recall:

$$\text{score} = \alpha_r \cdot \text{recency} + \alpha_i \cdot \text{importance} + \alpha_v \cdot \text{relevance}$$

- **recency**: exponential decay, $\gamma^{\Delta t}$ with $\gamma = 0.995$
  and $\Delta t$ in "hours" since the memory was last accessed.
- **importance**: scored 1-10 at write time (here we set it by hand, like a tag).
- **relevance**: cosine similarity between the memory and the current query.

Each term is min-max normalized to [0, 1] before summing (equal weights here,
$\alpha_r = \alpha_i = \alpha_v = 1$).


In [ ]:
GAMMA = 0.995

memories = [
    {"text": "Airflow incident: `daily_ingest` DAG failed, manual retry needed",
     "importance": 7, "hours_ago": 2},
    {"text": "The user said they prefer concise answers, code first",
     "importance": 6, "hours_ago": 240},
    {"text": "Apple pie recipe shared in the #random channel",
     "importance": 1, "hours_ago": 20},
    {"text": "The `daily_ingest` DAG had already failed last month for the same reason",
     "importance": 5, "hours_ago": 720},
]

query = "the airflow job crashed again, what should I do?"
q_vec = toy_embed(query)


def min_max(values):
    lo, hi = min(values), max(values)
    if hi == lo:
        return [1.0 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]


recencies = [GAMMA ** m["hours_ago"] for m in memories]
importances = [m["importance"] for m in memories]
relevances = [cosine_similarity(q_vec, toy_embed(m["text"])) for m in memories]

r_n, i_n, v_n = min_max(recencies), min_max(importances), min_max(relevances)
scores = [r + i + v for r, i, v in zip(r_n, i_n, v_n)]

ranked = sorted(zip(memories, scores, recencies, importances, relevances),
                 key=lambda x: x[1], reverse=True)

print(f"{'score':>6}  {'recency':>8}  {'import.':>8}  {'relevance':>9}   memory")
for m, s, rec, imp, rel in ranked:
    print(f"{s:6.2f}  {rec:8.3f}  {imp:8d}  {rel:9.3f}   {m['text']}")


Notice: the memory "already failed last month" (high relevance but
very low recency) and "prefers concise answers" (medium recency,
low relevance but notable importance) compete for 2nd/3rd place — this is
exactly the kind of trade-off this score makes explicit rather than implicit.
Change `GAMMA`, the `hours_ago` values, or the $\alpha$ weights to see the effect.


## 3. LangGraph — short-term memory (checkpointer)

We build a minimal single-node graph, and observe what the
`checkpointer` + `thread_id` provide: per-conversation persistence, and
strict isolation between threads.


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver  # canonical name; "MemorySaver" remains an alias
from langchain_core.messages import AIMessage, HumanMessage


class _EchoLLM:
    """Toy, deterministic model to run the demo without an API key."""

    def invoke(self, messages):
        last_user = next((m.content for m in reversed(messages)
                           if isinstance(m, HumanMessage)), "")
        seen_names = [w for m in messages if isinstance(m, HumanMessage)
                      for w in m.content.split() if w.istitle()]
        if "name" in last_user.lower():
            reply = f"[mock-llm] you told me your name is: {seen_names[-1] if seen_names else '???'}"
        else:
            reply = f"[mock-llm] got it: {last_user!r}"
        return AIMessage(content=reply)


if HAS_ANTHROPIC:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)
else:
    llm = _EchoLLM()


def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)
graph.add_edge(START, "call_model")

checkpointer = InMemorySaver()  # dev only — PostgresSaver in production
app = graph.compile(checkpointer=checkpointer)


In [ ]:
config_a = {"configurable": {"thread_id": "user-42-session-A"}}

app.invoke({"messages": [HumanMessage(content="Hello, I'm Alex")]}, config=config_a)
result = app.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=config_a)
print("Thread A ->", result["messages"][-1].content)

config_b = {"configurable": {"thread_id": "user-42-session-B"}}
result_b = app.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=config_b)
print("Thread B (new thread_id) ->", result_b["messages"][-1].content)


Thread A "remembers" the name (same `thread_id`, chained
checkpoints); thread B knows nothing — it has never seen a checkpoint for that
`thread_id`. This is the whole mechanism of LangGraph short-term memory.


In [ ]:
# Time travel: list thread A's checkpoint history, and go back to an earlier one
history = list(app.get_state_history(config_a))
print(f"{len(history)} checkpoints recorded for thread A")
for snap in history:
    n_msgs = len(snap.values.get("messages", []))
    print(f"  checkpoint {snap.config['configurable']['checkpoint_id'][:8]}...  "
          f"({n_msgs} messages, next={snap.next})")

# Resume exactly from a specific earlier checkpoint:
if len(history) >= 2:
    earlier_checkpoint_id = history[-1].config["configurable"]["checkpoint_id"]
    earlier_config = {**config_a, "configurable": {**config_a["configurable"],
                                                     "checkpoint_id": earlier_checkpoint_id}}
    earlier_state = app.get_state(earlier_config)
    print("\nState at the first checkpoint:", [m.content for m in earlier_state.values["messages"]])


## 4. LangGraph — long-term memory (`Store`)

The `Store` is independent of threads: you write/read namespaced
memories to it (typically per user), with semantic search if you
provide it an embedding function.


In [ ]:
from langgraph.store.memory import InMemoryStore


def embed_fn(texts):
    return [toy_embed(t).tolist() for t in texts]


store = InMemoryStore(index={"embed": embed_fn, "dims": 256})

user_id = "user-42"
namespace = (user_id, "memories")

store.put(namespace, str(uuid.uuid4()), {"fact": "Prefers concise answers, code first"})
store.put(namespace, str(uuid.uuid4()), {"fact": "Data engineering team, uses Airflow"})
store.put(namespace, str(uuid.uuid4()), {"fact": "Doesn't like emojis in responses"})

hits = store.search(namespace, query="how should I format my responses for this user?")
for h in hits:
    print(f"  score={h.score:.3f}  {h.value}")


Each result carries a similarity score — this is the same
"cosine" mechanism from section 1, applied this time to real memories
that have been persisted, scoped by `namespace` (here, per user).


In [ ]:
# Memory managed by the agent itself (langmem) — requires a real tool-calling model
if HAS_ANTHROPIC:
    from langmem import create_manage_memory_tool, create_search_memory_tool
    from langgraph.prebuilt import create_react_agent

    memory_agent = create_react_agent(
        "anthropic:claude-haiku-4-5-20251001",
        tools=[
            create_manage_memory_tool(namespace=("memories",)),
            create_search_memory_tool(namespace=("memories",)),
        ],
        store=store,
    )
    out = memory_agent.invoke({"messages": [
        HumanMessage(content="Remember that I work on healthcare data pipelines, "
                              "so be especially careful about confidentiality.")
    ]})
    print(out["messages"][-1].content)
else:
    print("ANTHROPIC_API_KEY missing -> 'langmem' demo (agent managing its own memory) skipped.")
    print("The store.put/search mechanism above, however, remains fully functional offline.")


## 5. DeepAgents — sub-agents, virtual filesystem, cross-session persistence

This section requires the `deepagents` package **and** a real
tool-calling model — it is therefore guarded by an import check and an
API key check, so it doesn't break the rest of the notebook if either is missing.


In [ ]:
try:
    from deepagents import create_deep_agent
    from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
    DEEPAGENTS_AVAILABLE = True
except ImportError:
    DEEPAGENTS_AVAILABLE = False
    print("Package 'deepagents' not installed -> `pip install deepagents` to run this section.")

RUN_DEEPAGENTS_DEMO = DEEPAGENTS_AVAILABLE and HAS_ANTHROPIC
if DEEPAGENTS_AVAILABLE and not HAS_ANTHROPIC:
    print("deepagents is installed, but ANTHROPIC_API_KEY is missing -> demo skipped.")


In [ ]:
if RUN_DEEPAGENTS_DEMO:
    cross_session_store = InMemoryStore()

    backend = lambda rt: CompositeBackend(
        default=StateBackend(rt),                          # ephemeral draft, scope = thread
        routes={"/memories/": StoreBackend(rt)},            # everything under /memories/ survives across sessions
    )

    agent = create_deep_agent(
        model="anthropic:claude-sonnet-5",
        backend=backend,
        store=cross_session_store,
    )

    thread_a = {"configurable": {"thread_id": "deepagents-demo-A"}}
    agent.invoke(
        {"messages": [HumanMessage(content=(
            "Summarize in 2 sentences the difference between short-term and long-term memory "
            "in LangGraph, and write that summary to /memories/notes.md"
        ))]},
        config=thread_a,
    )

    # New thread, SAME store -> the file under /memories/ should be visible
    thread_b = {"configurable": {"thread_id": "deepagents-demo-B"}}
    result_b = agent.invoke(
        {"messages": [HumanMessage(content="Read /memories/notes.md and quote it.")]},
        config=thread_b,
    )
    print(result_b["messages"][-1].content)
else:
    print("DeepAgents demo not run (see message above). "
          "In the meantime, review the pattern in the slides (part 2, backends).")


**Watch live if you have an API key:** thread B never "saw"
thread A's conversation — yet it retrieves the file, because
`/memories/` is routed to a `StoreBackend` that shares the same `store` as
thread A. A file written *outside* of `/memories/` (i.e. on the
default `StateBackend`) would not have survived the thread change —
try it as an exercise.


## Going further

- Theory slides: `docs/slides/agent-memory-part1-theory.html`
- Practice slides: `docs/slides/agent-memory-part2-practice.html`
- CoALA — <https://arxiv.org/abs/2309.02427>
- Generative Agents — <https://arxiv.org/abs/2304.03442>
- MemGPT — <https://arxiv.org/abs/2310.08560>
- LangGraph — persistence & memory (docs.langchain.com/oss/python/langgraph)
- DeepAgents — <https://github.com/langchain-ai/deepagents>
- Anthropic, *Effective context engineering for AI agents* (anthropic.com/engineering)
- Chroma Research, *Context Rot* (2025) — <https://www.trychroma.com/research/context-rot>
